In [1]:
from pathlib import Path
import subprocess, os

REPO_HINT = Path("/Users/mkrishan/Documents/boolearn-main")  # set None for public release

def find_repo_root():
    candidates = []
    if REPO_HINT is not None:
        candidates.append(Path(REPO_HINT).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates += [cwd] + list(cwd.parents)
    home = Path.home()
    candidates += [home/"boolearn-main", home/"Documents"/"boolearn-main", home/"Desktop"/"boolearn-main"]
    for p in candidates:
        if (p/"boolearn"/"src"/"makefile").exists():
            return p
    raise RuntimeError(f"Could not locate repo root from {cwd}. Set REPO_HINT.")

ROOT = find_repo_root()
SRC  = ROOT / "boolearn" / "src"
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("✅ ROOT:", ROOT)
print("✅ SRC:", SRC)
print("✅ DATA:", DATA)
print("✅ RESULTS:", RESULTS)

subprocess.run(["make","all"], cwd=str(SRC), check=True)
os.environ["PATH"] = f"{SRC}:{os.environ.get('PATH','')}"
print("✅ Built + PATH updated")

✅ ROOT: /Users/mkrishan/Documents/boolearn-main
✅ SRC: /Users/mkrishan/Documents/boolearn-main/boolearn/src
✅ DATA: /Users/mkrishan/Documents/boolearn-main/data
✅ RESULTS: /Users/mkrishan/Documents/boolearn-main/results
make: Nothing to be done for `all'.
✅ Built + PATH updated


In [3]:
from pathlib import Path

train   = SRC / "train"
layered = SRC / "layered"
assert train.exists() and layered.exists()

mult_dir = DATA / "mult"
dat = mult_dir / "2x2.dat"
assert dat.exists(), dat

print("train:", train)
print("layered:", layered)
print("data:", dat)

train: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train
layered: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered
data: /Users/mkrishan/Documents/boolearn-main/data/mult/2x2.dat


In [7]:
import subprocess
from pathlib import Path

OUT = RESULTS / "01_multiplier_2x2"
NET_DIR = OUT / "nets"
RUN_DIR = OUT / "runs"
NET_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

# .dat is in data/
mult_dat_dir = DATA / "mult"
dat = mult_dat_dir / "2x2.dat"
assert dat.exists(), dat

# .wth is in boolearn/
mult_wth_dir = ROOT / "boolearn" / "mult"
wth = mult_wth_dir / "2x2.wth"
assert wth.exists(), wth

net = NET_DIR / "2x2.net"
if not net.exists():
    cmd = [str(layered), "0.5", str(wth), str(net)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(NET_DIR), check=False)

assert net.exists() and net.stat().st_size > 0
print("✅ net:", net, "| bytes:", net.stat().st_size)
with open(net) as f:
    print("Header:", f.readline().strip())

Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered 0.5 /Users/mkrishan/Documents/boolearn-main/boolearn/mult/2x2.wth /Users/mkrishan/Documents/boolearn-main/results/01_multiplier_2x2/nets/2x2.net
✅ net: /Users/mkrishan/Documents/boolearn-main/results/01_multiplier_2x2/nets/2x2.net | bytes: 1238
Header: 14  5  4  49


In [9]:
import subprocess, time, re, select
from pathlib import Path

run_id = "mult2x2"
gap_path = RUN_DIR / f"{run_id}.gap"

# Use all 16 items (no test set => test% will show 0.000)
items  = "16"
btfn   = "3.0"
beta   = "0.2"
gam    = "1e-3"
epochs = "40"
itermax= "50000"
gapstop= "0.01"
trials = "1"

cmd = [str(train), str(net), str(dat),
       items, btfn, beta, gam, epochs, itermax, gapstop, trials, run_id]

print("Running:", " ".join(cmd))
print("RUN_DIR:", RUN_DIR)
print("Watching:", gap_path.name)

proc = subprocess.Popen(
    cmd,
    cwd=str(RUN_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

last_gap_lines = 0
printed = False

try:
    while True:
        # discard stdout without blocking (train is usually quiet)
        if proc.stdout is not None:
            r, _, _ = select.select([proc.stdout], [], [], 0.0)
            if r:
                _ = proc.stdout.readline()

        # print only new gap lines (log steps)
        if gap_path.exists():
            lines = [ln.strip() for ln in gap_path.read_text().splitlines() if ln.strip()]

            if lines and not printed:
                print("\niter\ttrain%\ttest%")
                printed = True

            if len(lines) > last_gap_lines:
                for ln in lines[last_gap_lines:]:
                    it = ln.split()[0]
                    pcts = re.findall(r"(\d+(?:\.\d+)?)\s*%", ln)
                    if len(pcts) >= 2:
                        tr, te = float(pcts[-2]), float(pcts[-1])
                        print(f"{it}\t{tr:.3f}\t{te:.3f}")
                last_gap_lines = len(lines)

        if proc.poll() is not None:
            break

        time.sleep(0.2)

finally:
    try:
        proc.kill()
    except Exception:
        pass

print("\nReturn code:", proc.returncode)
print("\nArtifacts:")
for p in sorted(RUN_DIR.glob(run_id + ".*")):
    print(" -", p.name, "|", p.stat().st_size, "bytes")

Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train /Users/mkrishan/Documents/boolearn-main/results/01_multiplier_2x2/nets/2x2.net /Users/mkrishan/Documents/boolearn-main/data/mult/2x2.dat 16 3.0 0.2 1e-3 40 50000 0.01 1 mult2x2
RUN_DIR: /Users/mkrishan/Documents/boolearn-main/results/01_multiplier_2x2/runs
Watching: mult2x2.gap

iter	train%	test%
2	56.250	0.000
3	62.500	0.000
4	62.500	0.000
5	64.062	0.000
6	67.188	0.000
7	65.625	0.000
8	65.625	0.000
9	65.625	0.000
12	65.625	0.000
15	67.188	0.000
20	68.750	0.000
26	73.438	0.000
34	75.000	0.000
45	81.250	0.000
58	85.938	0.000
76	85.938	0.000
100	90.625	0.000
131	89.062	0.000
171	90.625	0.000
224	96.875	0.000
294	96.875	0.000
385	95.312	0.000
504	95.312	0.000
660	90.625	0.000
865	96.875	0.000
1134	93.750	0.000
1486	93.750	0.000
1947	85.938	0.000
2552	92.188	0.000
3344	85.938	0.000
4383	96.875	0.000
5744	93.750	0.000
7528	96.875	0.000
9866	95.312	0.000
12931	92.188	0.000
16947	87.500	0.000
22210	93.750	0.000
29109	96.875	0

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim

# quick loader for boolearn .dat
def load_dat_bits_bool(path: Path):
    with open(path) as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    it = iter(lines)
    N_total = int(next(it))
    _, IN = map(int, next(it).split())
    _, OUT = map(int, next(it).split())
    X, Y = [], []
    for _ in range(N_total):
        X.append(list(map(int, next(it).split())))
        Y.append(list(map(int, next(it).split())))
    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    return X, Y, IN, OUT

X, Y, IN, OUT = load_dat_bits_bool(dat)
assert IN == 4 and OUT == 4 and len(X) == 16

# tiny MLP
model = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),
    nn.Linear(16, 4)
)
opt = optim.AdamW(model.parameters(), lr=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

for step in range(2000):
    opt.zero_grad()
    loss = loss_fn(model(X), Y)
    loss.backward()
    opt.step()

with torch.no_grad():
    probs = torch.sigmoid(model(X))
    pred = (probs >= 0.5).float()
    bit_acc = (pred == Y).float().mean().item() * 100
    exact_acc = (pred.eq(Y).all(dim=1)).float().mean().item() * 100

print(f"[GD baseline] bitwise={bit_acc:.2f}% exact={exact_acc:.2f}%")

[GD baseline] bitwise=100.00% exact=100.00%
